> 在线讲义：[第 1 章 · 运控的三层与 PD 目标接口：策略不直接给力矩](https://wikieden.github.io/robot-motion-rl/lessons/01-layers-pd.html) · 本书《机器人运控与强化学习》

# 第 1 章 · 运控的三层与 PD 目标接口：策略不直接给力矩

配套讲义：仓库里 `robot-motion-rl/notes/lessons/01-layers-pd.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（增益、负载、角度都是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 三层 | 1000 / 50 | `pd_steps_per_policy_step()` | 20 |
| 第二步 PD | $K_p(q^*-q)-K_d\dot q$ | `pd_torque(40, 2, 0.5, 0.3, 1.0)` | 6.0；加饱和 5.0 |
| 第三步 稳态误差 | 负载 / Kp | `steady_state_error(4, kp)` | 0.100、0.020 |
| 第三步 仿真 | 1 kHz 积分 2 s | `simulate_joint(kp, kd, 0.5, 4)` | 0.400、0.480 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `pd_torque`**：PD 目标接口：τ = Kp(q* − q) − Kd·q̇（目标速度取 0），可选力矩饱和 ±τ_max。

In [ ]:
def pd_torque(kp, kd, q_target, q, qd, tau_max=None):
    """PD 目标接口：τ = Kp(q* − q) − Kd·q̇（目标速度取 0），可选力矩饱和 ±τ_max。"""
    tau = kp * (q_target - q) - kd * qd
    return tau if tau_max is None else float(np.clip(tau, -tau_max, tau_max))

**函数 `pd_steps_per_policy_step`**：策略每输出一次设定值，底层 PD 要跑几次。

In [ ]:
def pd_steps_per_policy_step(pd_hz=1000, policy_hz=50):
    """策略每输出一次设定值，底层 PD 要跑几次。"""
    return pd_hz // policy_hz

**函数 `steady_state_error`**：恒定负载（如重力）下 PD 的稳态误差：Kp·e = 负载 → e = 负载 / Kp。

In [ ]:
def steady_state_error(load, kp):
    """恒定负载（如重力）下 PD 的稳态误差：Kp·e = 负载 → e = 负载 / Kp。"""
    return load / kp

**函数 `simulate_joint`**：1 自由度关节：I·q̈ = τ_PD − 负载，1 kHz 积分，返回最后的角度。

In [ ]:
def simulate_joint(kp, kd, q_target, load, inertia=0.1, seconds=2.0, dt=0.001):
    """1 自由度关节：I·q̈ = τ_PD − 负载，1 kHz 积分，返回最后的角度。"""
    q, qd = 0.0, 0.0
    for _ in range(int(seconds / dt)):
        tau = pd_torque(kp, kd, q_target, q, qd)
        qdd = (tau - load) / inertia
        qd += qdd * dt
        q += qd * dt
    return q

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 三层频率")
print("1 kHz PD / 50 Hz 策略：每个策略动作对应", pd_steps_per_policy_step(), "次 PD")

In [ ]:
print("\n第二步 PD 力矩")
print("Kp=40, Kd=2, q*=0.5, q=0.3, q̇=1.0 → τ =", pd_torque(40, 2, 0.5, 0.3, 1.0), "N·m")
print("加饱和 τ_max=5 →", pd_torque(40, 2, 0.5, 0.3, 1.0, tau_max=5), "N·m")

In [ ]:
print("\n第三步 低增益：设定值是力矩代理")
for kp in (40, 200):
    e = steady_state_error(4.0, kp)
    q_end = simulate_joint(kp, 2 * np.sqrt(kp * 0.1), 0.5, 4.0)
    print(f"Kp={kp}: 稳态误差 = 4/{kp} = {e:.3f} rad；仿真最后 q = {q_end:.3f}（目标 0.5）")
print("想停在 0.5、负载 4 N·m、Kp=40：设定值要给 0.5 + 0.1 =", 0.5 + steady_state_error(4.0, 40))

## 3. 代码里才看得到的

- 仿真里 Kd 取了临界阻尼 $2\sqrt{K_p I}$（I = 0.1），这样 2 秒内能停稳；讲义第二步的 Kd = 2 只是算一次力矩用的例子。
- 仿真最后的角度和「目标 − 负载/Kp」差不到 0.002，测试按这个容差核对。

## 4. 自己改一改

- 把 `simulate_joint` 的 kd 改成临界阻尼的 0.1 倍，看最后的角度还稳不稳（打印中间过程）。
- 把 dt 改成 0.02（相当于 PD 也只跑 50 Hz），看会不会发散。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 高层想、中层定、底层出力：VLA 几赫兹、策略五十赫兹、PD 一千赫兹；
> 弹簧拉向目标、阻尼刹住速度，力矩到顶就截平；
> 增益调低关节软，负载压出一截差，目标角变成力矩钮。